## The Verlet Algorithm
The final numerical method for the integration of ODEs that we will implement in this class is the *Verlet Algorithm*, which is often stated explicitly in term of kinematic state variables and acceleration:

$$x_{n+1} = x_n + v_n\Delta t + \frac{1}{2}a_n\Delta t^2$$
$$v_{n+1} = v_n + (a_{n+1}+a_n)\Delta t $$

To make use of this algorithm, you will have to manage current acceleration, $a_n$, and next acceleration $a_{n+1}$ in order to resolve the second line. Note that it is assumed that acceleration is a function of position, so that to find the next acceleration you evaluate $a_{n+1}(x_{n+1})$.

Write a function to do the Verlet update. It should accept a function to compute force as well as a datastructure *of your choice* that stores $x_n$, $v_n$, and $a_n$. At the end of the function's execution the data structure should be updated so that $x_n$, $v_n$, and $a_n$ are now $x_{n+1}$, $v_{n+1}$, and $a_{n+1}$. Consult the text and give careful thought to how the Verlet update will handle periodic boundary conditions on the force calculation. 

In [ ]:
import numpy as np

# L is the side length of the cubic simulation box
L = None  

def verlet_step(state, force_func, dt):
    """
    Parameters:
        state      : dict with keys x, y, z, vx, vy, vz, ax, ay, az
        force_func : callable(x, y, z) returning (fx, fy, fz)
        dt         : time step size
    """
    # update positions using equation above
    state['x'] = state['x'] + state['vx'] * dt + 0.5 * state['ax'] * dt**2
    state['y'] = state['y'] + state['vy'] * dt + 0.5 * state['ay'] * dt**2
    state['z'] = state['z'] + state['vz'] * dt + 0.5 * state['az'] * dt**2

    # apply periodic boundary conditions to the new positions.
    # If a particle leaves the box on one side, it re-enters on the other.
    state['x'] = state['x'] % L
    state['y'] = state['y'] % L
    state['z'] = state['z'] % L

    # compute forces at the new positions to get n+1
    ax_new, ay_new, az_new = force_func(state['x'], state['y'], state['z'])

    # update velocities using
    state['vx'] = state['vx'] + 0.5 * (ax_new + state['ax']) * dt
    state['vy'] = state['vy'] + 0.5 * (ay_new + state['ay']) * dt
    state['vz'] = state['vz'] + 0.5 * (az_new + state['az']) * dt

    # store
    state['ax'] = ax_new
    state['ay'] = ay_new
    state['az'] = az_new

## Distances
No matter the force we use, it will depend on the distances between particles. You've already worked some of this out wen you wrote `n-body`. I'd invite you to redo that function again, and try and do better in terms of performance. At core, the distances will be distances in the $x$, $y$, and $z$ directions, eg $\delta x_{i,j} = x_i = x_j$. The most naive way to write this would be:

```python
for i in range (N): # N is number of particles:
    for j in range(i+1,N):
        dx = x[i] - x[j]
        dy = y[i] - y[j]
        dz = z[i] - z[j]
        # Update periodic BC for image
        # Store the results in an array
```

But this is slow. Based on the data strucure you devised in the previous section, write the code to do this "all-pairs" loop as quickly as you can. I see two possibilities:
* Keep the explicit loops and use `numba` to 'compile' the code
* Use `numpy` to manage arrays of values that are manipulated to find all pairs of differences.

I have done both. I have also tried something called `jax` and discovered that the overhead of setting up the code makes run times slower than either of the above cases, at least for ~100 particles.

You will have to make accomodations here for the periodic boundary conditions, which is done by subtracting the length of the box from the co-ordinate differences the half length of the box.

So that is easy to compare implementations, call this function `distance_matrix` and accept imputs of x,y,z. Run PBC on the distances you compute inside the function to implement the *image method* discussed in the text.

In [ ]:
def distance_matrix(x, y, z):
    """
    Parameters:
        x, y, z : 1D arrays of particle positions, length N

    Returns:
        dx, dy, dz : NxN arrays of pairwise separation components
        r          : NxN array of pairwise distances
    """
    # broadcasting to get all pair differences at once
    # entry [i,j] = x[i] - x[j]
    dx = x[:, np.newaxis] - x[np.newaxis, :]
    dy = y[:, np.newaxis] - y[np.newaxis, :]
    dz = z[:, np.newaxis] - z[np.newaxis, :]

    # minimum image convention
    # wraps separations into [-L/2, L/2]
    dx = dx - L * np.round(dx / L)
    dy = dy - L * np.round(dy / L)
    dz = dz - L * np.round(dz / L)

    # scalar distances
    r = np.sqrt(dx**2 + dy**2 + dz**2)

    return dx, dy, dz, r

## Improved Lennard-Jones
Finally, using the `distance_matrix` function you just wrote, change the Lennard Jones force function you previously wrote to compute the forces for $N$ particles. Integrate the call to the function that determines forces (Lennard-Jones) into the Verlet algorithm you wrote in the first part of this assignment. Upon completion you should have code that can be wired together to:

1. You call a Verlet algorithm to evolve the state (positions and velocities) from $y_n$ to $y_{n+1}$. Acceleration is not part of the state, but should be maintained in whatever data structure stores state because it is needed in Verlet ($a_n$ and $a_{n+1}$)
2. After finding the next position, $x_{n+1}$,and updating the periodic boundary conditions, the Verlet algorithm calls a force function, Lennard-Jones in this case, to get the $a_{n+1}$.
3. The Lennard-Jones force function calls the `distance_matrix` function to find the distances between all pairs of particles. That function uses the *minimum-image approximation* (Figure 8.3) to ensure distances reflect the periodic boundary conditions. The `distance_matrix` is the most time consuming portion of the process, and deserves careful attention for optimization.
4. Lennard-Jones uses the distances returned by `distance_matrix` to compute the forces and return them.
5. Verlet uses the returned forces to find the next acceleration, $a_{n+1}$ and get the next velocity state, $v_{n+1}$ using the Verlet algorithm.

Do this and have this code base prepared to solve real problems for Tuesday, March 31.

In [ ]:
def lennard_jones_force(x, y, z, epsilon=1.0, sigma=1.0):
    """
    Parameters:
        x, y, z : 1D arrays of particle positions, length N
        epsilon : LJ energy parameter (default 1 for reduced units)
        sigma   : LJ length parameter (default 1 for reduced units)

    Returns:
        fx, fy, fz : 1D arrays of net force on each particle
    """
    # get all pairwise separations with minimum image convention
    dx, dy, dz, r = distance_matrix(x, y, z)

    # set diagonal to inf so self-interaction is zero
    np.fill_diagonal(r, np.inf)

    # (sigma/r)^6 and (sigma/r)^12
    sr6 = (sigma / r)**6
    sr12 = sr6**2

    # force prefactor, divided by r for projecting onto components
    f_over_r = 24.0 * epsilon * (2.0 * sr12 - sr6) / r**2

    # sum over j to get net force on each particle i
    fx = np.sum(f_over_r * dx, axis=1)
    fy = np.sum(f_over_r * dy, axis=1)
    fz = np.sum(f_over_r * dz, axis=1)

    return fx, fy, fz